# 🔍 NLP Desktop Search Engine — Inverted Index (50,000+ Files)
**Lab Assignment | NLP Course**

This notebook implements a full desktop keyword search engine step-by-step:

| Step | Task |
|------|------|
| 1 | Imports, Configuration & File Registry |
| 2 | Sample Data Creation (100 `.txt` files) |
| 3 | Preprocessing Pipeline |
| 4 | Verification — Indexed Files |
| 5 | Build Inverted Index |
| 6 | Index Persistence (Save / Load) |
| 7 | Vector Variants — Boolean · Count · TF-IDF |
| 8 | Similarity Measures — Dot Product · Cosine |
| 9 | Query Engine (Top-k Retrieval) |
| 10 | **Bonus** — Phrase & Proximity Queries |

In [1]:
import os
import re
import pickle
import math
import random
from pathlib import Path
from collections import defaultdict

# ─── Configuration ────────────────────────────────────────────────────────────
LOCAL_CORPUS_PATH = "./my_test_docs"   # folder containing .txt files
INDEX_DIR         = "./search_index"   # folder to persist the index

# ─── Data Structures ──────────────────────────────────────────────────────────
# Requirement 3A: Maintain docID ↔ file path mapping
doc_id_map    = {}   # full_path  → docID
id_to_path_map = {}  # docID      → full_path

def build_file_registry(root_path):
    """Recursively finds .txt files and assigns a unique integer docID."""
    doc_id_map.clear()
    id_to_path_map.clear()
    current_id = 0
    # Requirement 3B: Recursive file discovery
    for root, dirs, files in os.walk(root_path):
        for file in sorted(files):                   # sorted for determinism
            if file.endswith(".txt"):
                full_path = os.path.join(root, file)
                doc_id_map[full_path]      = current_id
                id_to_path_map[current_id] = full_path
                current_id += 1
    print(f"Total files indexed: {len(id_to_path_map)}")

## Step 2 — Create Sample Data
Creates **15 hand-crafted** + **85 auto-generated** `.txt` files across topic sub-folders, then builds the file registry.

In [2]:
# ─── Step 2A: Hand-crafted sample documents ───────────────────────────────────
SAMPLE_DIR = LOCAL_CORPUS_PATH
os.makedirs(SAMPLE_DIR, exist_ok=True)

sample_docs = [
    ("cricket.txt",
     "Pakistan cricket team won the match against India. The cricket players performed well on the ground. Babar Azam scored a brilliant century in the cricket tournament."),
    ("punjab_university.txt",
     "Punjab University is located in Lahore Pakistan. The university offers many degree programs. Students from Punjab University excel in research and academics every year."),
    ("nlp.txt",
     "Natural language processing NLP is a field of artificial intelligence. NLP techniques include tokenization stemming and lemmatization. Text classification is a common NLP task."),
    ("search_engine.txt",
     "Search engines use inverted index to find documents quickly. Google and Bing are popular search engines. Keyword search is fundamental to information retrieval systems."),
    ("python_programming.txt",
     "Python is a popular programming language for data science. Python is used for machine learning web development and automation. The language has a simple and readable syntax."),
    ("machine_learning.txt",
     "Machine learning algorithms learn patterns from data automatically. Supervised learning uses labeled data for training models. Neural networks are powerful machine learning models."),
    ("deep_learning.txt",
     "Deep learning uses multi-layer neural networks for feature extraction. Convolutional networks handle image recognition tasks. Recurrent networks process sequential data well."),
    ("data_science.txt",
     "Data science combines statistics programming and domain knowledge. Data scientists analyze large datasets to extract useful insights. Python and R are favourite languages."),
    ("lahore.txt",
     "Lahore is the capital city of Punjab province in Pakistan. The city is famous for its Mughal architecture and street food. Lahore has Badshahi Mosque and Lahore Fort."),
    ("islamabad.txt",
     "Islamabad is the federal capital of Pakistan. The city is known for modern infrastructure and lush greenery. Faisal Mosque is one of the largest mosques in the world."),
    ("karachi.txt",
     "Karachi is the largest city and economic hub of Pakistan. It is the major seaport of the country. The city has a diverse population and many industrial zones."),
    ("information_retrieval.txt",
     "Information retrieval systems index documents for fast search. Inverted index stores term to document mappings efficiently. Cosine similarity is used to rank documents by relevance."),
    ("vector_space.txt",
     "Vector space model represents documents as numerical vectors. TF-IDF is a popular weighting scheme in information retrieval. Cosine similarity measures relevance between document and query."),
    ("boolean_retrieval.txt",
     "Boolean retrieval uses binary weights for query terms. A term is either present or absent in a document. AND OR NOT operators are used to combine query terms."),
    ("tfidf.txt",
     "TF-IDF stands for term frequency inverse document frequency. It rewards terms frequent in a document but rare in the corpus. TF-IDF vectors improve search ranking quality significantly."),
]

for filename, content in sample_docs:
    filepath = os.path.join(SAMPLE_DIR, filename)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

# ─── Step 2B: Auto-generate 85 random documents in sub-folders ───────────────
topics = ["science", "history", "sports", "technology", "health", "education"]
words_pool = {
    "science":     ["research","experiment","laboratory","hypothesis","theory",
                    "discovery","biology","chemistry","physics","molecule","scientist","data"],
    "history":     ["ancient","civilization","empire","war","treaty","dynasty",
                    "king","battle","culture","artifact","museum","heritage"],
    "sports":      ["player","team","match","tournament","champion","league",
                    "coach","training","victory","stadium","cricket","football"],
    "technology":  ["computer","software","algorithm","network","cloud","server",
                    "database","internet","programming","hardware","system","code"],
    "health":      ["doctor","hospital","medicine","treatment","patient","disease",
                    "vaccine","nutrition","diagnosis","therapy","clinic","surgery"],
    "education":   ["student","teacher","curriculum","degree","university","exam",
                    "scholarship","lecture","library","research","campus","course"],
}

random.seed(42)
for i in range(85):
    topic = topics[i % len(topics)]
    subdir = os.path.join(SAMPLE_DIR, topic)
    os.makedirs(subdir, exist_ok=True)
    filename = f"{topic}_doc_{i + 1:03d}.txt"
    pool = words_pool[topic]
    sentences = []
    for _ in range(random.randint(3, 5)):
        words = random.choices(pool, k=random.randint(8, 12))
        sentences.append(" ".join(words).capitalize())
    content = ". ".join(sentences) + "."
    with open(os.path.join(subdir, filename), "w", encoding="utf-8") as f:
        f.write(content)

total_files = sum(len(fs) for _, _, fs in os.walk(SAMPLE_DIR))
print(f"✅ Sample documents created in '{SAMPLE_DIR}'")
print(f"   Total .txt files: {total_files}")

# ─── Build the file registry now that files exist ─────────────────────────────
build_file_registry(LOCAL_CORPUS_PATH)

✅ Sample documents created in './my_test_docs'
   Total .txt files: 102
Total files indexed: 102


## Step 3 — Preprocessing Pipeline
- Lowercase → Tokenize → Remove punctuation  
- *(Optional extensions: stopword removal, stemming/lemmatization)*

In [3]:
def preprocess_text(text):
    """
    Implements the Preprocessing Pipeline:
    1. Lowercasing (Requirement 3B) [cite: 32]
    2. Tokenization & Punctuation Removal (Requirement 3B) 
    """
    # Convert to lowercase
    text = text.lower()
    
    # Use regex to keep only alphanumeric characters (removes punctuation)
    # This splits the string into a list of words (tokens)
    tokens = re.findall(r'\b\w+\b', text)
    
    return tokens

# Test the preprocessor
sample_text = "NLP Lab: Building a Search Engine! (50,000+ files)"
print(f"Original: {sample_text}")
print(f"Tokens: {preprocess_text(sample_text)}")

Original: NLP Lab: Building a Search Engine! (50,000+ files)
Tokens: ['nlp', 'lab', 'building', 'a', 'search', 'engine', '50', '000', 'files']


## Step 4 — Verification: Indexed File Registry
Displays all indexed `docID → file path` mappings after building the registry.

In [4]:
# --- Verification: Display Indexed File Names ---
import os

print(f"{'DocID':<7} | {'File Name':<25} | {'Full Path'}")
print("-" * 80)

if not id_to_path_map:
    print("No files have been indexed yet. Please run Step 1.")
else:
    for doc_id, full_path in id_to_path_map.items():
        # Extract only the file name from the full path for clarity
        file_name = os.path.basename(full_path)
        print(f"{doc_id:<7} | {file_name:<25} | {full_path}")

print("-" * 80)
print(f"Total Files Indexed: {len(id_to_path_map)}")

DocID   | File Name                 | Full Path
--------------------------------------------------------------------------------
0       | A university is a high-level educat.txt | ./my_test_docs\A university is a high-level educat.txt
1       | School.txt                | ./my_test_docs\School.txt
2       | boolean_retrieval.txt     | ./my_test_docs\boolean_retrieval.txt
3       | cricket.txt               | ./my_test_docs\cricket.txt
4       | data_science.txt          | ./my_test_docs\data_science.txt
5       | deep_learning.txt         | ./my_test_docs\deep_learning.txt
6       | information_retrieval.txt | ./my_test_docs\information_retrieval.txt
7       | islamabad.txt             | ./my_test_docs\islamabad.txt
8       | karachi.txt               | ./my_test_docs\karachi.txt
9       | lahore.txt                | ./my_test_docs\lahore.txt
10      | machine_learning.txt      | ./my_test_docs\machine_learning.txt
11      | nlp.txt                   | ./my_test_docs\nlp.txt
12      |

## Step 5 — Build Inverted Index
For every document, tokenize → count term frequencies → update the posting list.

**Structure stored:**
```
inverted_index[term][docID] = tf       # term frequency
doc_freq[term]              = df       # number of docs containing term
doc_lengths[docID]          = length   # total tokens in doc
```

In [5]:
def build_inverted_index(id_to_path_map, preprocess_fn):
    """
    Requirement 3C — Inverted Index Construction
    Returns:
        inverted_index : dict  { term: {docID: tf} }
        doc_freq       : dict  { term: df }
        doc_lengths    : dict  { docID: total_tokens }
    """
    inv_idx  = defaultdict(lambda: defaultdict(int))  # term → {docID: tf}
    df_map   = defaultdict(int)                       # term → df
    lengths  = {}                                     # docID → token count

    total = len(id_to_path_map)
    for doc_id, path in id_to_path_map.items():
        try:
            with open(path, "r", encoding="utf-8", errors="ignore") as fh:
                text = fh.read()
        except Exception:
            continue

        tokens = preprocess_fn(text)
        lengths[doc_id] = len(tokens)

        # Count term frequencies for this document
        tf_map = defaultdict(int)
        for token in tokens:
            tf_map[token] += 1

        # Merge into the global inverted index
        for term, tf in tf_map.items():
            inv_idx[term][doc_id] = tf
            df_map[term] += 1

        if (doc_id + 1) % 25 == 0 or (doc_id + 1) == total:
            print(f"  Indexed {doc_id + 1:>4}/{total} documents…", end="\r")

    print(f"\n✅ Indexing complete.")
    # Convert nested defaultdicts to plain dicts
    inv_idx = {t: dict(postings) for t, postings in inv_idx.items()}
    return inv_idx, dict(df_map), lengths


# Build the inverted index
inverted_index, doc_freq, doc_lengths = build_inverted_index(
    id_to_path_map, preprocess_text
)

N = len(id_to_path_map)   # total number of documents
print(f"Unique terms   : {len(inverted_index)}")
print(f"Total documents: {N}")

# Preview first 5 postings
print("\n── Index Preview (first 5 terms) ──────────────────────")
for term in list(inverted_index)[:5]:
    print(f"  '{term}' → postings={inverted_index[term]}  df={doc_freq[term]}")

  Indexed  102/102 documents…
✅ Indexing complete.
Unique terms   : 584
Total documents: 102

── Index Preview (first 5 terms) ──────────────────────
  'a' → postings={0: 11, 1: 6, 2: 2, 3: 1, 8: 1, 11: 2, 13: 2, 15: 1, 16: 1}  df=9
  'university' → postings={0: 8, 12: 3, 17: 1, 18: 6, 19: 1, 20: 5, 21: 10, 22: 4, 23: 4, 24: 2, 25: 3, 26: 2, 27: 4, 28: 2, 29: 6, 30: 2}  df=16
  'is' → postings={0: 2, 1: 1, 2: 1, 6: 1, 7: 3, 8: 2, 9: 2, 11: 2, 12: 1, 13: 2, 14: 1, 16: 1}  df=12
  'high' → postings={0: 3, 1: 1}  df=2
  'level' → postings={0: 2, 1: 2}  df=2


## Step 6 — Index Persistence (Save / Load)
Saves the inverted index to disk so it can be reloaded later **without rebuilding**.  
Files saved: `inverted_index.pkl`, `doc_maps.pkl`, `doc_freq.pkl`, `doc_lengths.pkl`

In [6]:
os.makedirs(INDEX_DIR, exist_ok=True)

IDX_FILE       = os.path.join(INDEX_DIR, "inverted_index.pkl")
DOCMAP_FILE    = os.path.join(INDEX_DIR, "doc_maps.pkl")
DOCFREQ_FILE   = os.path.join(INDEX_DIR, "doc_freq.pkl")
DOCLENGTH_FILE = os.path.join(INDEX_DIR, "doc_lengths.pkl")


def save_index():
    """Persist all index structures to disk using pickle."""
    with open(IDX_FILE,       "wb") as f: pickle.dump(inverted_index, f)
    with open(DOCMAP_FILE,    "wb") as f: pickle.dump((doc_id_map, id_to_path_map), f)
    with open(DOCFREQ_FILE,   "wb") as f: pickle.dump(doc_freq, f)
    with open(DOCLENGTH_FILE, "wb") as f: pickle.dump(doc_lengths, f)
    print(f"✅ Index saved to '{INDEX_DIR}/'")
    for fpath in [IDX_FILE, DOCMAP_FILE, DOCFREQ_FILE, DOCLENGTH_FILE]:
        size_kb = os.path.getsize(fpath) / 1024
        print(f"   {os.path.basename(fpath):<28}  {size_kb:6.1f} KB")


def load_index():
    """Reload index from disk — no rebuilding needed."""
    global inverted_index, doc_id_map, id_to_path_map, doc_freq, doc_lengths, N
    with open(IDX_FILE,       "rb") as f: inverted_index = pickle.load(f)
    with open(DOCMAP_FILE,    "rb") as f: doc_id_map, id_to_path_map = pickle.load(f)
    with open(DOCFREQ_FILE,   "rb") as f: doc_freq = pickle.load(f)
    with open(DOCLENGTH_FILE, "rb") as f: doc_lengths = pickle.load(f)
    N = len(id_to_path_map)
    print(f"✅ Index loaded — {len(inverted_index)} terms, {N} documents")


# Save NOW
save_index()

# To reload in a fresh session, simply call: load_index()

✅ Index saved to './search_index/'
   inverted_index.pkl              14.4 KB
   doc_maps.pkl                     5.0 KB
   doc_freq.pkl                     6.8 KB
   doc_lengths.pkl                  0.4 KB


## Step 7 — Vector Variants
Three weighting schemes for documents **and** queries:

| Mode | Formula |
|------|---------|
| **Boolean** | $w = 1$ if term present, else $0$ |
| **Count** | $w = tf(t, d)$ — raw term frequency |
| **TF-IDF** | $w = tf(t,d) \times \bigl(\log\frac{N+1}{df(t)+1}+1\bigr)$ |

In [7]:
# ── Weight helpers ────────────────────────────────────────────────────────────

def get_boolean_weight(term, doc_id):
    """Binary: 1 if term in doc, else 0."""
    return 1 if (term in inverted_index and doc_id in inverted_index[term]) else 0


def get_count_weight(term, doc_id):
    """Raw term frequency."""
    return inverted_index.get(term, {}).get(doc_id, 0)


def compute_idf(term):
    """Smoothed IDF: log((N+1) / (df+1)) + 1"""
    df = doc_freq.get(term, 0)
    return math.log((N + 1) / (df + 1)) + 1


def get_tfidf_weight(term, doc_id):
    """TF-IDF weight: tf * idf."""
    tf = get_count_weight(term, doc_id)
    return tf * compute_idf(term) if tf > 0 else 0.0


# ── Vector builders ───────────────────────────────────────────────────────────

def build_doc_vector(doc_id, terms, mode="tfidf"):
    """Return a weight vector for `doc_id` over the given term list."""
    if mode == "boolean":
        return [float(get_boolean_weight(t, doc_id)) for t in terms]
    elif mode == "count":
        return [float(get_count_weight(t, doc_id))   for t in terms]
    else:
        return [get_tfidf_weight(t, doc_id)           for t in terms]


def build_query_vector(query_tokens, terms, mode="tfidf"):
    """Return a weight vector for query tokens over the given term list."""
    query_tf = defaultdict(int)
    for t in query_tokens:
        query_tf[t] += 1
    vec = []
    for term in terms:
        if term not in query_tf:
            vec.append(0.0)
        elif mode == "boolean":
            vec.append(1.0)
        elif mode == "count":
            vec.append(float(query_tf[term]))
        else:  # tfidf
            vec.append(query_tf[term] * compute_idf(term))
    return vec


# ── Demo: show weights for a few terms in doc 0 ───────────────────────────────
demo_terms = ["cricket", "university", "python", "search", "learning"]
demo_doc   = 0
print(f"Document {demo_doc}: {os.path.basename(id_to_path_map.get(demo_doc, 'N/A'))}\n")
print(f"{'Term':<15} {'Boolean':>9} {'Count':>7} {'TF-IDF':>10}")
print("─" * 45)
for t in demo_terms:
    b  = get_boolean_weight(t, demo_doc)
    c  = get_count_weight(t, demo_doc)
    tf = get_tfidf_weight(t, demo_doc)
    print(f"{t:<15} {b:>9} {c:>7} {tf:>10.4f}")

Document 0: A university is a high-level educat.txt

Term              Boolean   Count     TF-IDF
─────────────────────────────────────────────
cricket                 0       0     0.0000
university              1       8    22.4121
python                  0       0     0.0000
search                  0       0     0.0000
learning                1       1     3.8430


## Step 8 — Similarity Measures

| Measure | Formula |
|---------|---------|
| **Inner Product** | $\text{score}(d,q) = \sum_t w(t,d)\cdot w(t,q)$ |
| **Cosine Similarity** | $\text{score}(d,q) = \dfrac{\sum_t w(t,d)\cdot w(t,q)}{\lVert d\rVert \cdot \lVert q\rVert}$ |

In [8]:
def dot_product(vec_a, vec_b):
    """Inner product (dot product) of two equal-length vectors."""
    return sum(a * b for a, b in zip(vec_a, vec_b))


def vector_norm(vec):
    """Euclidean (L2) norm."""
    return math.sqrt(sum(x * x for x in vec))


def cosine_similarity(vec_a, vec_b):
    """
    Cosine similarity between two vectors.
    Returns 0 if either vector is the zero vector.
    """
    norm_a, norm_b = vector_norm(vec_a), vector_norm(vec_b)
    if norm_a == 0.0 or norm_b == 0.0:
        return 0.0
    return dot_product(vec_a, vec_b) / (norm_a * norm_b)


# ── Demo: compare two documents ───────────────────────────────────────────────
doc_a, doc_b     = 0, 1
# Use the first 20 unique index terms as the shared vector space
sample_terms     = list(inverted_index.keys())[:20]

vec_a = build_doc_vector(doc_a, sample_terms, mode="tfidf")
vec_b = build_doc_vector(doc_b, sample_terms, mode="tfidf")

print(f"Doc {doc_a} : {os.path.basename(id_to_path_map.get(doc_a, ''))}")
print(f"Doc {doc_b} : {os.path.basename(id_to_path_map.get(doc_b, ''))}")
print()
print(f"  Dot Product       : {dot_product(vec_a, vec_b):.4f}")
print(f"  Cosine Similarity : {cosine_similarity(vec_a, vec_b):.4f}")

Doc 0 : A university is a high-level educat.txt
Doc 1 : School.txt

  Dot Product       : 5181.3130
  Cosine Similarity : 0.8742


## Step 9 — Query Engine (Top-k Retrieval)
Supports:
- Any of the **3 vector modes**: `"boolean"` · `"count"` · `"tfidf"`
- Either **similarity measure**: `"cosine"` · `"dot"`
- Configurable **top-k** (default = 10)

Output shows: **Rank · Score · File Path**

In [9]:
def search(query, mode="tfidf", similarity="cosine", top_k=10):
    """
    Keyword search engine.

    Parameters
    ----------
    query      : str   — free-text query
    mode       : str   — 'boolean' | 'count' | 'tfidf'
    similarity : str   — 'cosine' | 'dot'
    top_k      : int   — number of ranked results to return
    """
    query_tokens = preprocess_text(query)
    if not query_tokens:
        print("⚠  Empty query after preprocessing.")
        return []

    # Candidate documents: union of posting lists for all query terms
    candidate_docs = set()
    for token in query_tokens:
        if token in inverted_index:
            candidate_docs.update(inverted_index[token].keys())

    if not candidate_docs:
        print(f"  No documents found for: '{query}'")
        return []

    query_terms = list(set(query_tokens))          # unique query terms
    q_vec = build_query_vector(query_tokens, query_terms, mode=mode)

    # Score every candidate
    scores = []
    for doc_id in candidate_docs:
        d_vec = build_doc_vector(doc_id, query_terms, mode=mode)
        score = cosine_similarity(d_vec, q_vec) if similarity == "cosine" \
                else dot_product(d_vec, q_vec)
        if score > 0.0:
            scores.append((score, doc_id))

    scores.sort(reverse=True)
    top_results = scores[:top_k]

    # ── Display ──────────────────────────────────────────────────────────────
    print(f"\n Query : '{query}'")
    print(f" Mode  : {mode.upper()}  |  Similarity : {similarity.upper()}  |  Top-{top_k}")
    print("─" * 78)
    print(f"{'Rank':<5} {'Score':>8}   {'File Path'}")
    print("─" * 78)
    results = []
    for rank, (score, doc_id) in enumerate(top_results, 1):
        path = id_to_path_map[doc_id]
        print(f"{rank:<5} {score:>8.4f}   {path}")
        results.append((rank, score, path))
    print("─" * 78)
    return results


# ═══════════════════════════════════════════════════════════════════════════════
# Run sample queries — try all 3 modes
# ═══════════════════════════════════════════════════════════════════════════════
search("cricket Pakistan", mode="tfidf",   similarity="cosine")
search("university education research",    mode="tfidf",   similarity="cosine")
search("python machine learning",          mode="count",   similarity="dot")
search("inverted index search engine",     mode="boolean", similarity="cosine")


 Query : 'cricket Pakistan'
 Mode  : TFIDF  |  Similarity : COSINE  |  Top-10
──────────────────────────────────────────────────────────────────────────────
Rank     Score   File Path
──────────────────────────────────────────────────────────────────────────────
1       0.8728   ./my_test_docs\cricket.txt
2       0.8020   ./my_test_docs\punjab_university.txt
3       0.8020   ./my_test_docs\lahore.txt
4       0.8020   ./my_test_docs\karachi.txt
5       0.8020   ./my_test_docs\islamabad.txt
6       0.5973   ./my_test_docs\sports\sports_doc_081.txt
7       0.5973   ./my_test_docs\sports\sports_doc_075.txt
8       0.5973   ./my_test_docs\sports\sports_doc_063.txt
9       0.5973   ./my_test_docs\sports\sports_doc_057.txt
10      0.5973   ./my_test_docs\sports\sports_doc_039.txt
──────────────────────────────────────────────────────────────────────────────

 Query : 'university education research'
 Mode  : TFIDF  |  Similarity : COSINE  |  Top-10
────────────────────────────────────────────

[(1, 0.8660254037844387, './my_test_docs\\search_engine.txt'),
 (2, 0.8660254037844387, './my_test_docs\\information_retrieval.txt'),
 (3, 0.5, './my_test_docs\\tfidf.txt')]

## Step 10 — ⭐ Bonus: Phrase & Proximity Queries

| Feature | Example | What it does |
|---------|---------|--------------|
| **Phrase Query** | `"Punjab University"` | Exact adjacent phrase match |
| **Proximity Query** | `Pakistan Cricket ^5` | Terms within *k* words of each other |

Requires a **positional inverted index**: `{ term: { docID: [pos1, pos2, …] } }`

In [10]:
# ── Build positional inverted index ───────────────────────────────────────────

def build_positional_index(id_to_path_map, preprocess_fn):
    """
    Positional index: { term: { docID: [pos0, pos1, …] } }
    Position = token offset within the document (0-based).
    """
    pos_idx = defaultdict(lambda: defaultdict(list))
    total   = len(id_to_path_map)

    for doc_id, path in id_to_path_map.items():
        try:
            with open(path, "r", encoding="utf-8", errors="ignore") as fh:
                text = fh.read()
        except Exception:
            continue
        tokens = preprocess_fn(text)
        for pos, token in enumerate(tokens):
            pos_idx[token][doc_id].append(pos)

        if (doc_id + 1) % 25 == 0 or (doc_id + 1) == total:
            print(f"  Positional index: {doc_id + 1}/{total}", end="\r")

    print("\n✅ Positional index built.")
    return {term: dict(docs) for term, docs in pos_idx.items()}


# ── Phrase Query ───────────────────────────────────────────────────────────────

def phrase_query(phrase, pos_idx, id_to_path_map):
    """
    Return documents that contain the exact consecutive phrase.
    Example: phrase_query('Punjab University', ...)
    """
    tokens = preprocess_text(phrase)
    if not tokens:
        return []

    # Documents must contain ALL tokens
    candidate_docs = set(pos_idx.get(tokens[0], {}).keys())
    for token in tokens[1:]:
        candidate_docs &= set(pos_idx.get(token, {}).keys())

    matching = []
    for doc_id in candidate_docs:
        positions_0 = pos_idx[tokens[0]][doc_id]
        # For each starting position, verify tokens[1], tokens[2], … follow consecutively
        for start in positions_0:
            if all(
                (start + offset) in pos_idx[tokens[offset]][doc_id]
                for offset in range(1, len(tokens))
            ):
                matching.append(doc_id)
                break   # no need to check other starts for this doc
    return matching


# ── Proximity Query ────────────────────────────────────────────────────────────

def proximity_query(term1, term2, window, pos_idx, id_to_path_map):
    """
    Return documents where term1 and term2 appear within `window` tokens.
    Example: proximity_query('Pakistan', 'cricket', 5, ...)
    """
    t1 = preprocess_text(term1)
    t2 = preprocess_text(term2)
    if not t1 or not t2:
        return []
    t1, t2 = t1[0], t2[0]

    common = set(pos_idx.get(t1, {}).keys()) & set(pos_idx.get(t2, {}).keys())
    matching = []
    for doc_id in common:
        p1_list = pos_idx[t1][doc_id]
        p2_list = pos_idx[t2][doc_id]
        if any(abs(p1 - p2) <= window for p1 in p1_list for p2 in p2_list):
            matching.append(doc_id)
    return matching


# ═══════════════════════════════════════════════════════════════════════════════
# Build positional index
# ═══════════════════════════════════════════════════════════════════════════════
positional_index = build_positional_index(id_to_path_map, preprocess_text)

# ── Test Phrase Query ─────────────────────────────────────────────────────────
phrase = "Punjab University"
p_results = phrase_query(phrase, positional_index, id_to_path_map)
print(f'\nPhrase Query  : "{phrase}"')
print(f"Matching docs : {len(p_results)}")
for doc_id in p_results:
    print(f"  DocID {doc_id:>3} → {id_to_path_map[doc_id]}")

# ── Test Proximity Query ──────────────────────────────────────────────────────
w1, w2, win = "Pakistan", "cricket", 5
pr_results = proximity_query(w1, w2, win, positional_index, id_to_path_map)
print(f'\nProximity Query: "{w1}" ^{win} "{w2}"')
print(f"Matching docs  : {len(pr_results)}")
for doc_id in pr_results:
    print(f"  DocID {doc_id:>3} → {id_to_path_map[doc_id]}")

  Positional index: 102/102
✅ Positional index built.

Phrase Query  : "Punjab University"
Matching docs : 1
  DocID  12 → ./my_test_docs\punjab_university.txt

Proximity Query: "Pakistan" ^5 "cricket"
Matching docs  : 1
  DocID   3 → ./my_test_docs\cricket.txt


## Step 11 — Generate 50,000+ Files (Proof of Indexing)
This cell generates **50,000 `.txt` files** across multiple sub-folders to satisfy the dataset requirement.  
Run this once — it will take a minute, then rebuild and save the full index.

In [ ]:
import time
import random

TARGET_FILES  = 50_000
CORPUS_ROOT   = LOCAL_CORPUS_PATH   # ./my_test_docs
BULK_DIR      = os.path.join(CORPUS_ROOT, "bulk_corpus")

# Rich vocabulary pools per topic for realistic content
TOPICS = {
    "science":    ["atom","molecule","electron","nucleus","reaction","theory","experiment",
                   "biology","chemistry","physics","radiation","energy","hypothesis","data",
                   "gravity","telescope","microscope","evolution","genetics","species"],
    "history":    ["king","emperor","dynasty","war","treaty","civilization","conquest",
                   "revolution","empire","colony","trade","navy","parliament","monarchy",
                   "artifact","heritage","ancient","medieval","renaissance","plague"],
    "sports":     ["player","team","match","tournament","goal","coach","stadium","trophy",
                   "championship","league","referee","penalty","wicket","bowler","batsman",
                   "swimmer","marathon","sprint","medal","olympics"],
    "technology": ["computer","software","hardware","algorithm","network","server","cloud",
                   "database","internet","programming","security","encryption","firewall",
                   "microchip","processor","bandwidth","protocol","framework","deployment","API"],
    "health":     ["doctor","hospital","medicine","patient","vaccine","diagnosis","therapy",
                   "surgery","nutrition","disease","symptom","antibiotic","pandemic","immune",
                   "cardiology","neurology","radiology","pharmacy","rehabilitation","wellness"],
    "education":  ["student","teacher","university","curriculum","lecture","degree","exam",
                   "scholarship","library","research","thesis","campus","faculty","semester",
                   "syllabus","assignment","laboratory","internship","graduation","accreditation"],
    "business":   ["company","revenue","profit","market","investment","strategy","merger",
                   "acquisition","dividend","equity","startup","entrepreneur","consultant",
                   "marketing","brand","customer","supply","logistics","contract","invoice"],
    "politics":   ["government","election","parliament","policy","democracy","president",
                   "minister","vote","constitution","legislation","party","opposition","senate",
                   "judiciary","diplomacy","treaty","sanction","coalition","referendum","sovereignty"],
    "environment":["climate","carbon","emissions","forest","ecosystem","biodiversity","pollution",
                   "renewable","solar","wind","recycling","conservation","habitat","glacier",
                   "ozone","drought","flood","sustainability","species","atmosphere"],
    "culture":    ["art","music","literature","film","poetry","theater","sculpture","painting",
                   "tradition","festival","language","heritage","cuisine","religion","philosophy",
                   "mythology","folklore","architecture","dance","ceremony"],
}

topic_list = list(TOPICS.keys())

def generate_random_document(topic, doc_num):
    """Create a realistic multi-sentence text for a given topic."""
    pool = TOPICS[topic]
    random.seed(doc_num * 7 + topic_list.index(topic))
    sentences = []
    for _ in range(random.randint(4, 8)):
        words = random.choices(pool, k=random.randint(10, 18))
        sentences.append(" ".join(words).capitalize() + ".")
    return " ".join(sentences)


print("⏳ Generating 50,000 .txt files — please wait…")
start_time = time.time()

total_created = 0
files_per_batch = 500          # 100 batches × 500 files = 50,000

for batch_idx in range(100):   # 100 batches
    topic = topic_list[batch_idx % len(topic_list)]
    batch_dir = os.path.join(BULK_DIR, topic, f"batch_{batch_idx:03d}")
    os.makedirs(batch_dir, exist_ok=True)

    for file_idx in range(files_per_batch):
        global_num = batch_idx * files_per_batch + file_idx
        filename   = f"doc_{global_num:06d}.txt"
        content    = generate_random_document(topic, global_num)
        with open(os.path.join(batch_dir, filename), "w", encoding="utf-8") as fh:
            fh.write(content)
        total_created += 1

    if (batch_idx + 1) % 10 == 0:
        elapsed = time.time() - start_time
        print(f"  {total_created:>6,} files created … ({elapsed:.1f}s elapsed)", end="\r")

elapsed = time.time() - start_time
print(f"\n✅ Generated {total_created:,} files in {elapsed:.1f} seconds")
print(f"   Location: {os.path.abspath(BULK_DIR)}")

# ── Rebuild the full index over ALL files (sample + bulk) ─────────────────────
print("\n⏳ Rebuilding file registry over all files…")
build_file_registry(LOCAL_CORPUS_PATH)

print("\n⏳ Rebuilding inverted index — this may take a few minutes…")
t0 = time.time()
inverted_index, doc_freq, doc_lengths = build_inverted_index(id_to_path_map, preprocess_text)
N  = len(id_to_path_map)
print(f"   Index built in {time.time() - t0:.1f}s")
print(f"   Total documents : {N:,}")
print(f"   Unique terms    : {len(inverted_index):,}")

# ── Save the full index ───────────────────────────────────────────────────────
save_index()

# ── Proof screenshot data ─────────────────────────────────────────────────────
print("\n" + "═" * 60)
print("  PROOF OF INDEXING")
print("═" * 60)
print(f"  Total .txt files indexed : {N:,}")
print(f"  Unique vocabulary terms  : {len(inverted_index):,}")
print(f"  Index folder             : {os.path.abspath(INDEX_DIR)}/")
for fname in ["inverted_index.pkl", "doc_maps.pkl", "doc_freq.pkl", "doc_lengths.pkl"]:
    fpath = os.path.join(INDEX_DIR, fname)
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"    {fname:<28}  {size_mb:.2f} MB")
print("═" * 60)